# 年別・コスト・非重複の検証

歴史的な実験コードです。現在の実行入口は `../09_confidence_nested.ipynb`。
元Notebookのセル番号は0始まりです。コードの個人フォルダ名は置換しています。独立実行は保証しません。
保存出力は `../../results/imported_20260907/`、監査は `../../docs/CONFIDENCE_AUDIT.md` を参照してください。


## 元のセル index 34


In [ ]:
# ============================================================
# 15分足10年
# HIGH CONFIDENCE ROBUSTNESS TEST
#
# 目的
# ------------------------------------------------------------
# 「RFのConfidenceが高いほど利益が大きい」
# という前回の結果が、
#
# ・年が変わっても再現するか
# ・BUY/SELL両方で再現するか
# ・取引コストを上げても残るか
#
# を時系列OOSで検証する。
#
# 改善点:
# 1. シグナル確定後、次の足OpenでEntry
# 2. 30分保有中の重複取引を禁止
# 3. 各年は、それ以前のデータだけで学習
# 4. Confidence閾値は固定して比較
# ============================================================


from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score


# ============================================================
# 1. 設定
# ============================================================

CSV_PATH = (
    Path.cwd()
    / "dukascopy_usdjpy"
    / "usdjpy_15m_2016_2026.csv"
)

# 15分足2本 = 30分保有
HORIZON_BARS = 2

# 最低3年程度は学習に使う
FIRST_TEST_YEAR = 2020

RANDOM_STATE = 42

N_ESTIMATORS = 200

# 前回見つかった領域を固定して検証
THRESHOLDS = [
    0.50,
    0.52,
    0.54,
    0.55,
    0.56,
    0.58,
    0.60,
    0.62,
    0.65,
]

# Confidence帯
CONF_BINS = [
    0.50,
    0.52,
    0.54,
    0.56,
    0.58,
    0.60,
    0.62,
    0.65,
    0.70,
    0.75,
    0.80,
    0.90,
    1.01,
]

CONF_LABELS = [
    "50-52",
    "52-54",
    "54-56",
    "56-58",
    "58-60",
    "60-62",
    "62-65",
    "65-70",
    "70-75",
    "75-80",
    "80-90",
    "90-100",
]

# コストStress Test
#
# 0.00002 = 0.002%
# 0.00004 = 0.004%
COST_LEVELS = [
    0.0,
    0.00002,
    0.00004,
    0.00006,
]


# ============================================================
# 2. 読み込み
# ============================================================

df = pd.read_csv(
    CSV_PATH,
    index_col=0,
    parse_dates=True,
)

df.index = pd.to_datetime(
    df.index,
    utc=True,
)

df = (
    df
    .sort_index()
    .copy()
)

df.columns = [
    c.lower()
    for c in df.columns
]


print("データ読み込み完了")
print("総行数:", len(df))
print(
    "期間:",
    df.index.min(),
    "→",
    df.index.max(),
)


# ============================================================
# 3. RSI
# ============================================================

def calculate_rsi(
    close,
    period=14,
):

    delta = close.diff()

    gain = delta.clip(
        lower=0
    )

    loss = -delta.clip(
        upper=0
    )

    avg_gain = (
        gain
        .rolling(period)
        .mean()
    )

    avg_loss = (
        loss
        .rolling(period)
        .mean()
    )

    rs = (
        avg_gain
        /
        avg_loss.replace(
            0,
            np.nan
        )
    )

    return (
        100
        -
        100
        /
        (
            1 + rs
        )
    )


# ============================================================
# 4. 特徴量
# ============================================================

def make_features(
    data,
):

    x = data.copy()

    # --------------------------------------------------------
    # Return
    # --------------------------------------------------------

    for n in [
        1,
        2,
        4,
        8,
        16,
    ]:

        x[
            f"return_{n}"
        ] = (
            x["close"]
            .pct_change(n)
        )

    # --------------------------------------------------------
    # Volatility
    # --------------------------------------------------------

    for n in [
        4,
        8,
        16,
        32,
    ]:

        x[
            f"vol_{n}"
        ] = (
            x["return_1"]
            .rolling(n)
            .std()
        )

    # --------------------------------------------------------
    # Moving Average
    # --------------------------------------------------------

    for period in [
        5,
        10,
        20,
        50,
        100,
    ]:

        ma = (
            x["close"]
            .rolling(period)
            .mean()
        )

        x[
            f"ma{period}_distance"
        ] = (
            x["close"]
            /
            ma
            - 1
        )

        x[
            f"ma{period}_slope"
        ] = (
            ma.pct_change()
        )

    # --------------------------------------------------------
    # Candle
    # --------------------------------------------------------

    candle_range = (
        x["high"]
        -
        x["low"]
    ).replace(
        0,
        np.nan
    )

    x["body"] = (
        x["close"]
        -
        x["open"]
    ) / candle_range

    x["upper_wick"] = (
        x["high"]
        -
        x[
            [
                "open",
                "close",
            ]
        ].max(
            axis=1
        )
    ) / candle_range

    x["lower_wick"] = (
        x[
            [
                "open",
                "close",
            ]
        ].min(
            axis=1
        )
        -
        x["low"]
    ) / candle_range

    x["range_pct"] = (
        x["high"]
        -
        x["low"]
    ) / x["close"]

    # --------------------------------------------------------
    # RSI
    # --------------------------------------------------------

    x["rsi14"] = (
        calculate_rsi(
            x["close"],
            14,
        )
        / 100
    )

    # --------------------------------------------------------
    # ATR
    # --------------------------------------------------------

    previous_close = (
        x["close"]
        .shift(1)
    )

    true_range = pd.concat(
        [
            x["high"]
            -
            x["low"],

            (
                x["high"]
                -
                previous_close
            ).abs(),

            (
                x["low"]
                -
                previous_close
            ).abs(),
        ],
        axis=1,
    ).max(
        axis=1
    )

    x["atr14"] = (
        true_range
        .rolling(14)
        .mean()
        /
        x["close"]
    )

    # --------------------------------------------------------
    # 高値 / 安値距離
    # --------------------------------------------------------

    high16 = (
        x["high"]
        .rolling(16)
        .max()
    )

    low16 = (
        x["low"]
        .rolling(16)
        .min()
    )

    x[
        "distance_high_16"
    ] = (
        high16
        -
        x["close"]
    ) / x["close"]

    x[
        "distance_low_16"
    ] = (
        x["close"]
        -
        low16
    ) / x["close"]

    # --------------------------------------------------------
    # 時間
    # --------------------------------------------------------

    hour = (
        x.index.hour
        +
        x.index.minute
        / 60
    )

    x["hour_sin"] = np.sin(
        2
        *
        np.pi
        *
        hour
        /
        24
    )

    x["hour_cos"] = np.cos(
        2
        *
        np.pi
        *
        hour
        /
        24
    )

    x["weekday"] = (
        x.index.dayofweek
        / 4
    )

    return x


data = make_features(
    df
)


# ============================================================
# 5. より現実的なEntry / Exit
#
# t:
# 15分足が確定
# ↓
# t+1:
# 次の足OpenでEntry
# ↓
# t+2:
# 2本後CloseでExit
# ============================================================

data[
    "entry_price"
] = (
    data["open"]
    .shift(-1)
)

data[
    "exit_price"
] = (
    data["close"]
    .shift(
        -HORIZON_BARS
    )
)

data[
    "future_return"
] = (
    data[
        "exit_price"
    ]
    /
    data[
        "entry_price"
    ]
    - 1
)

data[
    "target"
] = (
    data[
        "future_return"
    ]
    > 0
).astype(int)


# ============================================================
# 6. 特徴量
# ============================================================

FEATURES = [

    "return_1",
    "return_2",
    "return_4",
    "return_8",
    "return_16",

    "vol_4",
    "vol_8",
    "vol_16",
    "vol_32",

    "ma5_distance",
    "ma5_slope",

    "ma10_distance",
    "ma10_slope",

    "ma20_distance",
    "ma20_slope",

    "ma50_distance",
    "ma50_slope",

    "ma100_distance",
    "ma100_slope",

    "body",
    "upper_wick",
    "lower_wick",
    "range_pct",

    "rsi14",
    "atr14",

    "distance_high_16",
    "distance_low_16",

    "hour_sin",
    "hour_cos",
    "weekday",
]


data = (
    data
    .replace(
        [
            np.inf,
            -np.inf,
        ],
        np.nan,
    )
    .dropna(
        subset=
            FEATURES
            +
            [
                "entry_price",
                "exit_price",
                "future_return",
            ]
    )
    .copy()
)


print()
print(
    "ML使用可能データ:",
    len(data)
)


# ============================================================
# 7. 各年を完全OOSで予測
#
# 例:
#
# 2020 Test
# → 2016-2019だけで学習
#
# 2021 Test
# → 2016-2020だけで学習
#
# ...
# ============================================================

oos_frames = []

year_model_results = []


available_years = sorted(
    data.index.year.unique()
)


for test_year in (
    available_years
):

    if (
        test_year
        <
        FIRST_TEST_YEAR
    ):

        continue

    train = (
        data.loc[
            data.index.year
            <
            test_year
        ]
        .copy()
    )

    test = (
        data.loc[
            data.index.year
            ==
            test_year
        ]
        .copy()
    )

    if (
        len(train) < 5000
        or
        len(test) < 100
    ):

        continue

    print()
    print(
        "===================================="
    )

    print(
        f"TEST YEAR: {test_year}"
    )

    print(
        "===================================="
    )

    print(
        "Train:",
        len(train)
    )

    print(
        "Test:",
        len(test)
    )

    model = RandomForestClassifier(

        n_estimators=
            N_ESTIMATORS,

        max_depth=
            8,

        min_samples_leaf=
            30,

        max_features=
            "sqrt",

        class_weight=
            "balanced",

        random_state=
            RANDOM_STATE,

        n_jobs=
            -1,
    )

    model.fit(
        train[
            FEATURES
        ],
        train[
            "target"
        ],
    )

    train_prob = (
        model.predict_proba(
            train[
                FEATURES
            ]
        )[:, 1]
    )

    test_prob = (
        model.predict_proba(
            test[
                FEATURES
            ]
        )[:, 1]
    )

    train_auc = (
        roc_auc_score(
            train[
                "target"
            ],
            train_prob,
        )
    )

    test_auc = (
        roc_auc_score(
            test[
                "target"
            ],
            test_prob,
        )
    )

    print(
        "Train AUC:",
        round(
            train_auc,
            4
        )
    )

    print(
        "Test AUC:",
        round(
            test_auc,
            4
        )
    )

    result = pd.DataFrame(
        index=
            test.index
    )

    result[
        "year"
    ] = (
        test_year
    )

    result[
        "p_up"
    ] = (
        test_prob
    )

    result[
        "p_down"
    ] = (
        1
        -
        test_prob
    )

    result[
        "confidence"
    ] = np.maximum(
        result[
            "p_up"
        ],
        result[
            "p_down"
        ]
    )

    result[
        "prediction"
    ] = np.where(
        result[
            "p_up"
        ]
        >= 0.5,
        1,
        -1,
    )

    result[
        "side"
    ] = np.where(
        result[
            "prediction"
        ]
        ==
        1,
        "BUY",
        "SELL",
    )

    result[
        "future_return"
    ] = (
        test[
            "future_return"
        ].values
    )

    result[
        "correct"
    ] = (
        (
            result[
                "future_return"
            ]
            *
            result[
                "prediction"
            ]
        )
        > 0
    )

    result[
        "gross_return"
    ] = (
        result[
            "future_return"
        ]
        *
        result[
            "prediction"
        ]
    )

    oos_frames.append(
        result
    )

    year_model_results.append(
        {
            "year":
                test_year,

            "train_rows":
                len(train),

            "test_rows":
                len(test),

            "train_auc":
                train_auc,

            "test_auc":
                test_auc,

            "auc_gap":
                train_auc
                -
                test_auc,
        }
    )


# ============================================================
# 8. OOS結合
# ============================================================

oos = (
    pd.concat(
        oos_frames
    )
    .sort_index()
)

model_year_df = (
    pd.DataFrame(
        year_model_results
    )
)


print()
print(
    "===================================="
)

print(
    "年間AUC"
)

print(
    "===================================="
)

print(
    model_year_df.to_string(
        index=False
    )
)


# ============================================================
# 9. 非重複取引
#
# 30分保有中は次の取引を禁止
# ============================================================

def select_non_overlapping(
    frame,
):

    if frame.empty:
        return frame.copy()

    frame = (
        frame
        .sort_index()
        .copy()
    )

    selected = []

    next_allowed_time = None

    hold_duration = (
        pd.Timedelta(
            minutes=30
        )
    )

    for time, row in (
        frame.iterrows()
    ):

        if (
            next_allowed_time
            is not None
            and
            time
            <
            next_allowed_time
        ):

            continue

        selected.append(
            row
        )

        next_allowed_time = (
            time
            +
            hold_duration
        )

    if not selected:

        return pd.DataFrame(
            columns=
                frame.columns
        )

    result = pd.DataFrame(
        selected
    )

    return result


# ============================================================
# 10. Stats
# ============================================================

def profit_factor(
    returns,
):

    r = np.asarray(
        returns,
        dtype=float
    )

    gain = (
        r[
            r > 0
        ].sum()
    )

    loss = (
        -r[
            r < 0
        ].sum()
    )

    if loss > 0:

        return (
            gain
            /
            loss
        )

    if gain > 0:

        return np.inf

    return np.nan


def calculate_stats(
    returns,
):

    r = np.asarray(
        returns,
        dtype=float
    )

    if len(r) == 0:

        return {
            "trades":
                0,

            "win_rate":
                np.nan,

            "avg_return":
                np.nan,

            "pf":
                np.nan,

            "max_dd":
                np.nan,

            "growth":
                np.nan,
        }

    equity = np.r_[
        1.0,
        np.cumprod(
            1 + r
        )
    ]

    peak = (
        np.maximum.accumulate(
            equity
        )
    )

    dd = (
        equity
        /
        peak
        - 1
    )

    return {

        "trades":
            len(r),

        "win_rate":
            (
                r > 0
            ).mean(),

        "avg_return":
            np.mean(
                r
            ),

        "pf":
            profit_factor(
                r
            ),

        "max_dd":
            np.min(
                dd
            ),

        "growth":
            equity[-1]
            - 1,
    }


# ============================================================
# 11. Confidence帯
# ============================================================

oos[
    "confidence_band"
] = pd.cut(
    oos[
        "confidence"
    ],
    bins=
        CONF_BINS,
    labels=
        CONF_LABELS,
    right=False,
)


band_rows = []


for band in (
    CONF_LABELS
):

    subset = (
        oos.loc[
            oos[
                "confidence_band"
            ]
            ==
            band
        ]
    )

    subset = (
        select_non_overlapping(
            subset
        )
    )

    if len(
        subset
    ) == 0:

        continue

    for cost in (
        COST_LEVELS
    ):

        returns = (
            subset[
                "gross_return"
            ]
            -
            cost
        )

        stats = (
            calculate_stats(
                returns
            )
        )

        band_rows.append(
            {
                "confidence_band":
                    band,

                "cost_pct":
                    cost
                    *
                    100,

                "trades":
                    stats[
                        "trades"
                    ],

                "accuracy":
                    subset[
                        "correct"
                    ].mean(),

                "avg_return":
                    stats[
                        "avg_return"
                    ],

                "pf":
                    stats[
                        "pf"
                    ],

                "max_dd":
                    stats[
                        "max_dd"
                    ],

                "growth":
                    stats[
                        "growth"
                    ],
            }
        )


band_results = pd.DataFrame(
    band_rows
)


# ============================================================
# 12. 閾値以上だけ取引
# ============================================================

threshold_rows = []


for threshold in (
    THRESHOLDS
):

    candidate = (
        oos.loc[
            oos[
                "confidence"
            ]
            >=
            threshold
        ]
    )

    candidate = (
        select_non_overlapping(
            candidate
        )
    )

    if candidate.empty:
        continue

    for cost in (
        COST_LEVELS
    ):

        returns = (
            candidate[
                "gross_return"
            ]
            -
            cost
        )

        stats = (
            calculate_stats(
                returns
            )
        )

        threshold_rows.append(
            {
                "threshold":
                    threshold,

                "cost_pct":
                    cost
                    *
                    100,

                "trades":
                    stats[
                        "trades"
                    ],

                "accuracy":
                    candidate[
                        "correct"
                    ].mean(),

                "avg_return":
                    stats[
                        "avg_return"
                    ],

                "pf":
                    stats[
                        "pf"
                    ],

                "max_dd":
                    stats[
                        "max_dd"
                    ],

                "growth":
                    stats[
                        "growth"
                    ],
            }
        )


threshold_results = (
    pd.DataFrame(
        threshold_rows
    )
)


# ============================================================
# 13. 年 × Threshold
# ============================================================

year_threshold_rows = []


for year in sorted(
    oos[
        "year"
    ].unique()
):

    year_data = (
        oos.loc[
            oos[
                "year"
            ]
            ==
            year
        ]
    )

    for threshold in (
        THRESHOLDS
    ):

        subset = (
            year_data.loc[
                year_data[
                    "confidence"
                ]
                >=
                threshold
            ]
        )

        subset = (
            select_non_overlapping(
                subset
            )
        )

        if subset.empty:
            continue

        # 基準コスト0.004%
        returns = (
            subset[
                "gross_return"
            ]
            -
            0.00004
        )

        stats = (
            calculate_stats(
                returns
            )
        )

        year_threshold_rows.append(
            {
                "year":
                    year,

                "threshold":
                    threshold,

                "trades":
                    stats[
                        "trades"
                    ],

                "accuracy":
                    subset[
                        "correct"
                    ].mean(),

                "avg_return":
                    stats[
                        "avg_return"
                    ],

                "pf":
                    stats[
                        "pf"
                    ],

                "growth":
                    stats[
                        "growth"
                    ],
            }
        )


year_threshold_results = (
    pd.DataFrame(
        year_threshold_rows
    )
)


# ============================================================
# 14. BUY / SELL × Threshold
# ============================================================

side_rows = []


for side in [
    "BUY",
    "SELL",
]:

    for threshold in (
        THRESHOLDS
    ):

        subset = (
            oos.loc[
                (
                    oos[
                        "side"
                    ]
                    ==
                    side
                )
                &
                (
                    oos[
                        "confidence"
                    ]
                    >=
                    threshold
                )
            ]
        )

        subset = (
            select_non_overlapping(
                subset
            )
        )

        if subset.empty:
            continue

        returns = (
            subset[
                "gross_return"
            ]
            -
            0.00004
        )

        stats = (
            calculate_stats(
                returns
            )
        )

        side_rows.append(
            {
                "side":
                    side,

                "threshold":
                    threshold,

                "trades":
                    stats[
                        "trades"
                    ],

                "accuracy":
                    subset[
                        "correct"
                    ].mean(),

                "avg_return":
                    stats[
                        "avg_return"
                    ],

                "pf":
                    stats[
                        "pf"
                    ],

                "growth":
                    stats[
                        "growth"
                    ],
            }
        )


side_results = pd.DataFrame(
    side_rows
)


# ============================================================
# 15. 年安定性まとめ
# ============================================================

stability_rows = []


for threshold in (
    THRESHOLDS
):

    subset = (
        year_threshold_results.loc[
            year_threshold_results[
                "threshold"
            ]
            ==
            threshold
        ]
    )

    if subset.empty:
        continue

    stability_rows.append(
        {
            "threshold":
                threshold,

            "years":
                len(
                    subset
                ),

            "positive_years":
                (
                    subset[
                        "avg_return"
                    ]
                    >
                    0
                ).sum(),

            "pf_above_1_years":
                (
                    subset[
                        "pf"
                    ]
                    >
                    1
                ).sum(),

            "median_pf":
                subset[
                    "pf"
                ].median(),

            "mean_pf":
                subset[
                    "pf"
                ].mean(),

            "median_return":
                subset[
                    "avg_return"
                ].median(),
        }
    )


stability_results = (
    pd.DataFrame(
        stability_rows
    )
)


# ============================================================
# 16. 表示
# ============================================================

print()
print(
    "===================================="
)

print(
    "閾値結果 Cost = 0.004%"
)

print(
    "===================================="
)


main_threshold = (
    threshold_results.loc[
        np.isclose(
            threshold_results[
                "cost_pct"
            ],
            0.004,
        )
    ]
    .copy()
)


main_show = (
    main_threshold.copy()
)


for col in [
    "threshold",
    "accuracy",
    "avg_return",
    "max_dd",
    "growth",
]:

    main_show[
        col
    ] *= 100


print(
    main_show.to_string(
        index=False
    )
)


print()
print(
    "===================================="
)

print(
    "年別 × Threshold"
)

print(
    "===================================="
)


year_show = (
    year_threshold_results
    .copy()
)


year_show[
    "threshold"
] *= 100


year_show[
    "accuracy"
] *= 100


year_show[
    "avg_return"
] *= 100


year_show[
    "growth"
] *= 100


print(
    year_show.to_string(
        index=False
    )
)


print()
print(
    "===================================="
)

print(
    "年安定性"
)

print(
    "===================================="
)


stability_show = (
    stability_results
    .copy()
)


stability_show[
    "threshold"
] *= 100


stability_show[
    "median_return"
] *= 100


print(
    stability_show.to_string(
        index=False
    )
)


print()
print(
    "===================================="
)

print(
    "BUY / SELL"
)

print(
    "===================================="
)


side_show = (
    side_results
    .copy()
)


side_show[
    "threshold"
] *= 100


side_show[
    "accuracy"
] *= 100


side_show[
    "avg_return"
] *= 100


side_show[
    "growth"
] *= 100


print(
    side_show.to_string(
        index=False
    )
)


# ============================================================
# 17. 2026だけ表示
# ============================================================

latest_year = (
    int(
        oos[
            "year"
        ].max()
    )
)


latest_results = (
    year_threshold_results.loc[
        year_threshold_results[
            "year"
        ]
        ==
        latest_year
    ]
    .copy()
)


latest_results[
    "threshold"
] *= 100


latest_results[
    "accuracy"
] *= 100


latest_results[
    "avg_return"
] *= 100


latest_results[
    "growth"
] *= 100


print()
print(
    "===================================="
)

print(
    f"{latest_year}年のみ"
)

print(
    "===================================="
)


print(
    latest_results.to_string(
        index=False
    )
)


# ============================================================
# 18. Cost Stress Test
# ============================================================

print()
print(
    "===================================="
)

print(
    "Cost Stress Test"
)

print(
    "===================================="
)


cost_show = (
    threshold_results
    .copy()
)


cost_show[
    "threshold"
] *= 100


cost_show[
    "avg_return"
] *= 100


cost_show[
    "growth"
] *= 100


print(
    cost_show[
        [
            "threshold",
            "cost_pct",
            "trades",
            "accuracy",
            "avg_return",
            "pf",
            "growth",
        ]
    ]
    .to_string(
        index=False
    )
)


# ============================================================
# 19. グラフ
# Threshold vs PF
# ============================================================

plt.figure(
    figsize=(
        9,
        5
    )
)


for cost in (
    COST_LEVELS
):

    temp = (
        threshold_results.loc[
            np.isclose(
                threshold_results[
                    "cost_pct"
                ],
                cost
                *
                100,
            )
        ]
    )

    plt.plot(
        temp[
            "threshold"
        ],
        temp[
            "pf"
        ],
        marker="o",
        label=
            f"Cost {cost * 100:.3f}%"
    )


plt.axhline(
    1,
    linewidth=1
)

plt.xlabel(
    "Minimum Confidence"
)

plt.ylabel(
    "Profit Factor"
)

plt.title(
    "Confidence vs Profit Factor"
)

plt.legend()

plt.tight_layout()

plt.show()


# ============================================================
# 20. Threshold vs accuracy
# ============================================================

plt.figure(
    figsize=(
        9,
        5
    )
)


plt.plot(
    main_threshold[
        "threshold"
    ],
    main_threshold[
        "accuracy"
    ],
    marker="o"
)


plt.xlabel(
    "Minimum Confidence"
)

plt.ylabel(
    "Accuracy"
)

plt.title(
    "Confidence vs OOS Accuracy"
)

plt.tight_layout()

plt.show()


# ============================================================
# 21. 年別PF heatmap風テーブル
# ============================================================

pf_pivot = (
    year_threshold_results
    .pivot(
        index=
            "year",

        columns=
            "threshold",

        values=
            "pf",
    )
)


print()
print(
    "===================================="
)

print(
    "YEAR × THRESHOLD PF"
)

print(
    "===================================="
)


print(
    pf_pivot.to_string()
)


# ============================================================
# 22. 保存
# ============================================================

OUTPUT_DIR = (
    Path.cwd()
    /
    "15m_confidence_robustness"
)

OUTPUT_DIR.mkdir(
    exist_ok=True
)


oos.to_csv(
    OUTPUT_DIR
    /
    "oos_predictions.csv"
)


model_year_df.to_csv(
    OUTPUT_DIR
    /
    "year_auc.csv",
    index=False
)


band_results.to_csv(
    OUTPUT_DIR
    /
    "confidence_bands.csv",
    index=False
)


threshold_results.to_csv(
    OUTPUT_DIR
    /
    "threshold_cost_stress.csv",
    index=False
)


year_threshold_results.to_csv(
    OUTPUT_DIR
    /
    "year_threshold_results.csv",
    index=False
)


side_results.to_csv(
    OUTPUT_DIR
    /
    "buy_sell_results.csv",
    index=False
)


stability_results.to_csv(
    OUTPUT_DIR
    /
    "stability_results.csv",
    index=False
)


print()
print(
    "===================================="
)

print(
    "検証終了"
)

print(
    "===================================="
)

print(
    OUTPUT_DIR.resolve()
)

print()

print(
    "特に見る場所:"
)

print(
    "1. 年安定性"
)

print(
    "2. 2026年のみ"
)

print(
    "3. YEAR × THRESHOLD PF"
)

print(
    "4. BUY / SELL"
)

print(
    "5. Cost Stress Test"
)